# Run and resume the full 36-condition study
Run from this package directory in your existing Blackwell-capable kernel.


In [ ]:
from pathlib import Path
import json, sys
assert Path("launch.py").exists(), "Open the notebook inside the extracted package folder"
from launch import run, train_all, benchmarks
C = json.loads(Path("config.json").read_text())
ROOT = Path(C["run_dir"])


## Main experiment
Runs one condition per subprocess. Completed jobs skip; partial jobs resume a hash-verified checkpoint. KL uses the online teacher; CE uses text labels only, but begins from the same pruned pretrained weights. Every student gets the same corpus/order/budget within a seed. Test results never choose checkpoints.

In [ ]:
from protocol import jobs
print("Conditions:", len(jobs(C)))
for j in jobs(C):
    print(j["id"])

## Launch
This is a long training job. Keep the Jupyter server alive according to your campus policy. No runtime promise. Each job saves initial/final validation, a recovery curve, an export, reload proof and final test/PIQA scores.

In [ ]:
train_all()

## Pause/resume controls
Use a SECOND notebook or terminal to create STOP while training is running. Do not start another training process. The commented cells below are intentional; uncomment only the action you want.

In [ ]:
# (ROOT / "STOP").touch()  # request pause at next optimizer boundary
# (ROOT / "STOP").unlink(missing_ok=True)  # after inspecting progress, allow resume

## Progress

In [ ]:
completed = sorted((ROOT / "metrics").glob("*_s*_*.json"))
print("Completed:", len(completed), "/", len(jobs(C)))
for p in completed:
    r = json.loads(p.read_text())
    print(p.stem, "PPL", r["test"]["perplexity"], "PIQA", r["piqa"]["accuracy"])